# **CNN Architecture**

![cnn architecture](https://miro.medium.com/v2/resize:fit:1400/format:webp/0*LeK_gmCf3DfO3gj_.jpeg)

There are different types of CNN Architectures:
- LENET
- AlexNET
- GoogleNET
- VggNET
- ResNET
- Inception

# **LENET**

## **Understanding the LeNet-5 Architecture**

LeNet-5 was designed by Yann LeCun, Leon Bottou, Yoshua Bengio, and Patrick Haffner in 1998. It is the grandfather of all modern Convolutional Neural Networks and was originally built to **read handwritten digits** on bank checks (the MNIST dataset).

## 1. Step-by-Step Layer Architecture

LeNet-5 takes a $32 \times 32$ grayscale image as input and processes it through the following sequence of layers:

### Input Layer
* **Dimensions:** $32 \times 32 \times 1$ (Grayscale)

### Layer C1: Convolutional Layer
* **Filters (Kernels):** 6 filters of size $5 \times 5$
* **Stride:** 1
* **Padding:** 0 (Valid padding)
* **Output Dimensions:** $28 \times 28 \times 6$
* **Activation:** Sigmoid (or Tanh)

### Layer S2: Average Pooling (Subsampling) Layer
* **Window Size:** $2 \times 2$
* **Stride:** 2
* **Output Dimensions:** $14 \times 14 \times 6$

### Layer C3: Convolutional Layer
* **Filters (Kernels):** 16 filters of size $5 \times 5$
* **Stride:** 1
* **Padding:** 0
* **Output Dimensions:** $10 \times 10 \times 16$
* **Activation:** Sigmoid (or Tanh)

### Layer S4: Average Pooling (Subsampling) Layer
* **Window Size:** $2 \times 2$
* **Stride:** 2
* **Output Dimensions:** $5 \times 5 \times 16$

### Layer C5: Convolutional / Dense Transition Layer
* **Filters (Kernels):** 120 filters of size $5 \times 5$
* **Stride:** 1
* **Output Dimensions:** $1 \times 1 \times 120$ (Acts like a flattened layer)
* **Activation:** Sigmoid (or Tanh)

### Layer F6: Fully Connected (Dense) Layer
* **Neurons:** 84 fully-connected units
* **Activation:** Sigmoid (or Tanh)

### Output Layer
* **Neurons:** 10 units (one for each digit 0-9)
* **Activation:** Softmax (originally Radial Basis Functions (RBF), but modernized to Softmax)



### Pros
* **Extremely Lightweight:** With only about 60,000 trainable parameters, it is incredibly fast to train and runs efficiently even on low-end CPUs.
* **Perfect for Simple Data:** Highly effective at reading digits, characters, and simple black-and-white shapes (such as license plates or postal zip codes).
* **Groundbreaking Foundations:** It established the standard sequence of "Convolution -> Pooling -> Convolution -> Pooling -> Dense" used in almost every modern CNN.

### Cons
* **Vanishing Gradients:** It relies on Sigmoid and Tanh activations. In deeper networks, these functions cause gradients to shrink to zero, stopping the model from learning.
* **Too Shallow for Complex Images:** With only two pooling layers and three convolutional layers, it cannot extract the complex color patterns needed for larger datasets like ImageNet (e.g., classifying cats, dogs, or cars).
* **Fuzzy Downsampling:** It uses Average Pooling, which blurs out sharp edge details compared to modern Max Pooling.

## 3. Modernized LeNet-5 Keras Code

Here is how you build a modernized version of LeNet-5 using Keras today (swapping Sigmoid/Tanh activations for ReLU, and Average Pooling for Max Pooling for better performance):

```python
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    # Input Layer (32x32 grayscale)
    layers.Input(shape=(32, 32, 1)),
    
    # Layer C1 (Convolution)
    layers.Conv2D(filters=6, kernel_size=(5, 5), strides=(1, 1), activation='relu'),
    
    # Layer S2 (Max Pooling replaces Average Pooling)
    layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    
    # Layer C3 (Convolution)
    layers.Conv2D(filters=16, kernel_size=(5, 5), strides=(1, 1), activation='relu'),
    
    # Layer S4 (Max Pooling)
    layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    
    # Flatten instead of C5 Convolution
    layers.Flatten(),
    
    # Layer F6 (Fully Connected / Dense)
    layers.Dense(units=120, activation='relu'),
    layers.Dense(units=84, activation='relu'),
    
    # Output Layer (10 classes for digit classification)
    layers.Dense(units=10, activation='softmax')
])

# Compile Model
model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()
```

---

# **Forward Propagation in CNN**

This guide details the step-by-step pipeline of a Convolutional Neural Network (CNN) and tracks exactly how image dimensions change mathematically as they pass through the model.

## 1. How a CNN Works: Step-by-Step

A CNN processes an image through a pipeline of feature extraction and classification:

#### **Step 1: Input Representation**
The image is fed into the network as a 3D grid of numbers: **Height x Width x Channels** (where channels are 3 for RGB color, or 1 for grayscale).

#### **Step 2: Convolution (Feature Extraction)**
Small filters (kernels) slide across the image. They perform element-wise multiplication and sum the results to create **Activation Maps**. Early layers detect simple features like edges, lines, and corners.

#### **Step 3: Activation (ReLU)**
The activation maps are passed through a ReLU activation function. This turns all negative values to zero, introducing non-linearity so the network can learn complex patterns.

#### **Step 4: Pooling (Downsampling)**
Pooling layers (like Max Pooling) slide across the activation maps to shrink their width and height. This discards redundant data, reduces computation, and makes the model resistant to minor shifts in the image.

#### **Step 5: Deeper Feature Learning**
Steps 2, 3, and 4 are repeated multiple times. Deeper layers combine the simple features (like edges) into complex features (like eyes, noses, wheels, or faces).

#### **Step 6: Flattening**
The final 3D stack of feature maps is squashed into a long 1D vector.

#### **Step 7: Classification (Fully Connected + Softmax)**
The 1D vector is passed through standard Dense layers, ending with a Softmax layer that outputs probability percentages for each class (e.g., 95% Dog, 3% Cat, 2% Bird).


---


## **How the Shape of an Image Changes in each stage ???**

An image's shape is represented as **(Height, Width, Channels)**. As it passes through the network, the Height and Width shrink, while the Channels (depth) increase.

#### A. How Convolution Changes Shape
When a kernel slides across an image, it shrinks the spatial dimensions unless you add padding. The formula to calculate the new height or width is:

$$\text{Output Size} = \frac{\text{Input Size} - \text{Kernel Size} + 2 \times \text{Padding}}{\text{Stride}} + 1$$

* **Padding = 'valid' (no padding):** The image shrinks. For example, if you pass a $32 \times 32$ image through a $5 \times 5$ kernel with a stride of 1, the new size is: $(32 - 5 + 1) = 28 \times 28$.
* **Padding = 'same' (zero padding):** The output height and width remain exactly the same as the input.
* **Channels (Depth):** The output depth is always **equal to the number of filters** you choose for that layer. If you use 16 filters, the output depth is 16.

#### B. How Pooling Changes Shape
Pooling shrinks the height and width, but **keeps the depth (channels) exactly the same**.

With a `pool_size=(2,2)` and `strides=(2,2)`, you divide both the height and width by **2** (since the window jumps 2 pixels at a time). For example, a shape of $(28, 28, 6)$ passed through a Max Pooling layer becomes $(14, 14, 6)$.

#### C. How Flattening Changes Shape
Flattening multiplies the height, width, and channels together to create a single 1D vector. For example, a final shape of $(5, 5, 16)$ is flattened into: $5 \times 5 \times 16 = \mathbf{400}$ nodes.

## 3. The LeNet-5 Shape Walkthrough

Let's trace exactly how the shape of a single MNIST digit changes as it goes through the LeNet-5 model:

* **Start (Input Image):**  
  Shape is **`(32, 32, 1)`**
* **Layer C1 (Conv2D with 6 filters of 5x5, valid padding):**  
  Height/Width shrink: $(32 - 5 + 1) = 28$. Channels become 6 (filters).  
  Shape becomes **`(28, 28, 6)`**
* **Layer S2 (MaxPooling2D of 2x2, stride 2):**  
  Height/Width cut in half: $28 / 2 = 14$. Channels stay 6.  
  Shape becomes **`(14, 14, 6)`**
* **Layer C3 (Conv2D with 16 filters of 5x5, valid padding):**  
  Height/Width shrink: $(14 - 5 + 1) = 10$. Channels become 16.  
  Shape becomes **`(10, 10, 16)`**
* **Layer S4 (MaxPooling2D of 2x2, stride 2):**  
  Height/Width cut in half: $10 / 2 = 5$. Channels stay 16.  
  Shape becomes **`(5, 5, 16)`**
* **Flatten Layer:**  
  Multiplies dimensions: $5 \times 5 \times 16 = 400$.  
  Shape becomes a 1D vector of size **`(400)`**
* **Dense Layers (Fully Connected):**  
  Shape goes from **`(400)`** $\rightarrow$ **`(120)`** $\rightarrow$ **`(84)`**
* **Output Layer (Softmax):**  
  Shape becomes **`(10)`** (Representing the probabilities for digits 0 through 9).

## **Why Did We Divided by 2 in Pooling?**

We divide the height and width of an image by 2 because the **stride size is 2**.

The stride tells the pooling window how many pixels to jump over at each step. Because `strides=(2,2)` means a jump of 2 pixels:

### If your width is 28 pixels:
* If you start at one end and jump 2 pixels at a time, you can only take exactly 14 steps before you run out of space.
  
  $$28 \text{ pixels} \div 2 \text{ pixel jump} = \mathbf{14} \text{ steps}$$
  
* Each step produces exactly 1 output pixel. Therefore, your new width is **14**.

### What if we changed the stride?
The number you divide by is **always equal to your stride**:

* **If we used `strides=(3,3)`:**  
  We would divide the height and width by 3 (e.g., a $27 \times 27$ image would become $9 \times 9$).
  
* **If we used `strides=(4,4)`:**  
  We would divide the height and width by 4 (e.g., a $28 \times 28$ image would become $7 \times 7$).

Because almost all standard CNN architectures (like LeNet-5, VGG, or ResNet) use a stride of 2 for pooling layers, we practically always divide the height and width by 2.

---

# How Padding Affects Pooling Output Shape

In a CNN pooling layer, the output shape is primarily determined by your stride. However, if your input dimension is an **odd number** (like 15 or 29) and cannot be divided perfectly, changing the padding will change the output size by **exactly +1 pixel**.

## 1. Even Input Dimensions (No Change)

If your input height or width is an even number (like 28), changing the padding does not affect the output. 

* **Input Size:** 28
* **Stride:** 2
* **Output size for `padding='valid'`:** $28 \div 2 = \mathbf{14}$
* **Output size for `padding='same'`:** $28 \div 2 = \mathbf{14}$

Both settings result in the exact same output size because 28 divides perfectly.

## 2. Odd Input Dimensions (Output Changes by +1)

If your input height or width is an odd number (like 15), the division is not clean ($15 \div 2 = 7.5$). Here is how Keras resolves this:

### Option A: `padding='valid'` (Default - No Padding)
The pooling window slides across the image, taking 7 steps of 2-pixel jumps. The 15th pixel at the very end is left over and cannot fit into a full 2x2 window, so Keras **discards (ignores) it**.
* **Math:** $\lfloor 15 \div 2 \rfloor = \mathbf{7}$ (Rounds down)
* **Output Size:** **7**

### Option B: `padding='same'` (Zero Padding)
To prevent discarding the 15th pixel, Keras automatically **adds 1 column of zeros** to the edge, padding the dimension to 16 pixels.
* **Math:** $15 + 1 \text{ (padding)} = 16 \text{ pixels}$. Then $16 \div 2 = \mathbf{8}$ steps. (Rounds up)
* **Output Size:** **8**

## 3. Visual Representation (15-Pixel Input)

Here is a 1D visualization of how the window slides across 15 pixels using a stride of 2:

### Under `padding='valid'` (No Padding):
```text
Step:     [ 1 ]   [ 2 ]   [ 3 ]   [ 4 ]   [ 5 ]   [ 6 ]   [ 7 ]  
Pixels:  (01 02) (03 04) (05 06) (07 08) (09 10) (11 12) (13 14) [15]  <-- Pixel 15 is discarded!
```
* **Result:** 7 successful steps $\rightarrow$ Output size is **7**.

### Under `padding='same'` (With Padding):
```text
Step:     [ 1 ]   [ 2 ]   [ 3 ]   [ 4 ]   [ 5 ]   [ 6 ]   [ 7 ]   [ 8 ]
Pixels:  (01 02) (03 04) (05 06) (07 08) (09 10) (11 12) (13 14) (15 [0]) <-- Zero added at the end!
```
* **Result:** 8 successful steps $\rightarrow$ Output size is **8**.

## 4. Summary Table (For Stride = 2, Pool Size = 2)

| Input Dimension | Output Size (`padding='valid'`) | Output Size (`padding='same'`) | Difference |
| :--- | :--- | :--- | :--- |
| **28 (Even)** | 14 | 14 | 0 |
| **29 (Odd)** | 14 (Rounds down) | 15 (Rounds up) | +1 |
| **30 (Even)** | 15 | 15 | 0 |
| **31 (Odd)** | 15 (Rounds down) | 16 (Rounds up) | +1 |

---

---

# **Why Do We Initialize CNN Kernels with Random Weights ?**

Historically, computer scientists in the 1990s did hand-craft specific kernels (like vertical or horizontal edge detectors) and locked them in place. In modern deep learning, however, we initialize all kernel numbers with random values before training begins.

## 1. The Three Reasons We Use Random Weights

### **Reason A: The Symmetry Problem**
If you have a layer with 32 filters, and you initialize all 32 of them to look like the exact same hand-crafted vertical edge detector:
* Every filter will calculate the exact same output during the forward pass.
* Every filter will receive the exact same gradient during backpropagation.
* Every filter will update to the exact same new values.

Because they are perfectly identical, all 32 filters will learn the exact same feature. You waste 31 filters doing duplicate work. Starting them with distinct random numbers breaks this symmetry, forcing Filter 1 to learn vertical edges, Filter 2 to learn horizontal edges, Filter 3 to learn curves, etc.

### **Reason B: We Don't Know the Optimal Features**
While a vertical edge detector is easy to design, what kind of $3 \times 3$ grid do you need to detect the texture of a sweater, a dog's whisker, or a cancer cell on an X-ray? Humans do not know the perfect math values for these features. By starting with random numbers, we let the optimizer discover the best patterns directly from the data.

### **Reason C: Deep Layers Learn Complex Shapes**
In the later layers of a CNN, the filters do not detect simple lines. They detect complex objects (like eyes, tires, or text characters). It is impossible for a human to design a static $3 \times 3$ grid of weights that naturally extracts "a cat's ear." The network must learn these complex shapes dynamically through gradient updates.

## 2. Using "Smart" Random Numbers (Initialization)

We do not choose completely blind random numbers (like values between $-100$ and $+100$) because that would cause gradients to explode or vanish. Instead, we use mathematically scaled random numbers:

* **He Normal (`he_normal`):** Used when the activation is **ReLU**. It draws weights from a normal distribution with a variance of $\frac{2}{n_{\text{in}}}$.
* **Glorot Normal (`glorot_normal` / Xavier):** Used when the activation is **Tanh** or **Sigmoid**.

## 3. The Kernel Life Cycle

During training, the weights in the kernel go through three distinct phases:

1. **Startup (Smart Random Numbers):**  
   Initialized with scaled random decimals using `he_normal` or `glorot_normal` to break symmetry and keep gradients stable.
2. **Training (Gradient Updates):**  
   Gradients are calculated during backpropagation, and weights are updated step-by-step:  
   
   $$w_{\text{new}} = w_{\text{old}} - \eta \cdot g$$
   
3. **Completion (Perfect Feature Detector):**  
   The weights settle into specific configurations (e.g., vertical edge detectors or curve detectors) optimized to solve the target task.
````_